In [ ]:
import numpy as np, pandas as pd, os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# Experiment C2 — Language-Design Loss

The PDF directly specifies a 3-term loss for constructing the universal language: corpus proximity (speaker-weighted KL), expressivity (coverage of concept space), and simplicity (description length). This experiment operationalises it: optimise a proto-vocabulary against this loss and show the trade-off curves. This is the closest the project gets to actually *constructing* the language.

In [ ]:
!pip install sentence-transformers wordfreq torch matplotlib pandas -q

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = ''
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
torch.manual_seed(42)

In [ ]:
!pip install sentence-transformers wordfreq -q
from sentence_transformers import SentenceTransformer
from wordfreq import top_n_list

# The PDF specifies an actual multi-term loss for the universal language:
#   L = L_proximity + L_expressivity + L_simplicity
#
# Term 1 (L_proximity):  KL divergence between the proto-language's token
#   distribution and the speaker-weighted real-language distribution.
#   "2x closer to English than Spanish if 2x more English speakers."
#
# Term 2 (L_expressivity): coverage of the interlingua centroid space.
#   A proto-vocabulary covers a concept if at least one proto-token is
#   close to the concept centroid. More coverage = more expressive.
#
# Term 3 (L_simplicity): description length of the vocabulary.
#   Penalises vocabularies with too many tokens or highly unequal distributions.
#
# We optimise a proto-vocabulary V = {v_1, ..., v_N} where each v_i is a
# learned 768-d unit vector, by gradient descent on L.

model = SentenceTransformer('LaBSE', device='cpu')

# ── Build concept space (same as Exp 9/10) ───────────────────────────────────
CONCEPTS = [
    'water','fire','earth','sky','wind','stone','river','mountain','forest','child',
    'elder','friend','enemy','time','life','death','peace','war','hope','dream',
    'change','love','fear','trust','joy','pain','give','take','speak','think',
    'find','lose','build','break','eat','feel','sleep','run','light','dark',
    'knife','hand','voice','home','silence','memory','freedom','beauty',
    'food','road','sky','tree','bird','fish','dog','sun','moon','star',
]
CONCEPTS = list(dict.fromkeys(CONCEPTS))  # dedup

SPEAKER_WEIGHTS = {'en':1500,'zh':1100,'hi':600,'es':560,'ar':380,'ru':260,'pt':260,'fr':280,'de':130,'ja':125}
print('Embedding concept centroids...')
concept_embs_dict = {}
for lang in SPEAKER_WEIGHTS:
    words_raw = top_n_list(lang, 8000)
    words_raw = [w for w in words_raw if len(w)>=3 and any(c.isalpha() for c in w)]
    words_for_lang = words_raw[:len(CONCEPTS)]
    # Use wordfreq top words as proxy; embed first len(CONCEPTS) content words
    concept_embs_dict[lang] = model.encode(words_for_lang, normalize_embeddings=True,
                                           show_progress_bar=False, batch_size=128)

# En embeddings as concept representations (same as Exp 9 centroid approach)
concept_embs_en = model.encode(CONCEPTS[:len(concept_embs_dict['en'])],
                               normalize_embeddings=True, show_progress_bar=False)
N_CONCEPTS = len(concept_embs_en)

# Speaker-weighted centroid matrix
total_w = sum(SPEAKER_WEIGHTS.values())
centroid_matrix = np.zeros((N_CONCEPTS, 768), dtype=np.float32)
for lang, w in SPEAKER_WEIGHTS.items():
    e = concept_embs_dict[lang][:N_CONCEPTS]
    centroid_matrix[:N_CONCEPTS] += (w/total_w) * e
centroid_matrix /= np.linalg.norm(centroid_matrix, axis=1, keepdims=True).clip(1e-9)
print('Concept centroids:', centroid_matrix.shape)

# Per-language token frequency distributions (proxy: uniform over top-1000 words)
# In production: use actual BPE token frequency distributions
lang_token_dists = {}
for lang, w in SPEAKER_WEIGHTS.items():
    # Simple proxy: log-uniform frequency over top words (Zipfian approximation)
    n_top = 1000
    ranks = np.arange(1, n_top+1, dtype=float)
    freqs = 1.0/ranks
    freqs /= freqs.sum()
    lang_token_dists[lang] = freqs

# Speaker-weighted target distribution
target_dist = np.zeros(1000, dtype=np.float64)
for lang, w in SPEAKER_WEIGHTS.items():
    target_dist += (w/total_w) * lang_token_dists[lang]
target_dist /= target_dist.sum()
print('Target token distribution computed (speaker-weighted Zipfian proxy)')

In [ ]:
# ── Proto-vocabulary optimisation ─────────────────────────────────────────────
PROTO_VOCAB_SIZES = [10, 20, 50, 100, 200]
LAM_PROX = 1.0   # weight on L_proximity
LAM_EXPR = 2.0   # weight on L_expressivity
LAM_SIMP = 0.5   # weight on L_simplicity
N_EPOCHS = 500

def language_design_loss(proto_vocab, centroid_matrix_t, target_dist_t, lam_prox, lam_expr, lam_simp):
    N_proto = proto_vocab.shape[0]
    proto_norm = F.normalize(proto_vocab, dim=-1)

    # L_proximity: proto-vocab should be close to the speaker-weighted centroid space
    # For each proto-token, find its closest centroid; average distance
    sims = proto_norm @ centroid_matrix_t.T  # (N_proto, N_concepts)
    nearest_centroid_sim = sims.max(dim=1).values
    L_prox = 1.0 - nearest_centroid_sim.mean()

    # L_expressivity: each concept should be close to at least one proto-token
    concept_nearest_sim = sims.max(dim=0).values
    # Coverage = fraction of concepts covered (sim > threshold)
    coverage = (concept_nearest_sim > 0.3).float().mean()
    L_expr = 1.0 - coverage  # minimise uncovered concepts

    # L_simplicity: penalise large vocab (description length)
    # Simple proxy: penalise N_proto/N_concepts ratio
    L_simp = torch.tensor(float(N_proto) / len(centroid_matrix_t))

    return lam_prox*L_prox + lam_expr*L_expr + lam_simp*L_simp, L_prox, L_expr, L_simp

centroid_t = torch.tensor(centroid_matrix, dtype=torch.float32)
target_dist_t = torch.tensor(target_dist[:100], dtype=torch.float32)  # use top-100

print('Optimising proto-vocabularies at different sizes...')
results = []
for N_proto in PROTO_VOCAB_SIZES:
    print('  N_proto={}...'.format(N_proto), end=' ')
    # Initialise proto-vocab by sampling from centroid space
    idx = np.random.default_rng(42).choice(N_CONCEPTS, min(N_proto, N_CONCEPTS), replace=False)
    proto_init = torch.tensor(centroid_matrix[idx], dtype=torch.float32)
    if N_proto > N_CONCEPTS:
        # Pad with random unit vectors
        extra = F.normalize(torch.randn(N_proto-N_CONCEPTS, 768), dim=-1)
        proto_init = torch.cat([proto_init, extra], 0)
    proto_vocab = nn.Parameter(proto_init.clone())
    opt = torch.optim.Adam([proto_vocab], lr=1e-2)

    losses = []
    for ep in range(N_EPOCHS):
        total, l_p, l_e, l_s = language_design_loss(proto_vocab, centroid_t, target_dist_t,
                                                      LAM_PROX, LAM_EXPR, LAM_SIMP)
        opt.zero_grad(); total.backward(); opt.step()
        if ep % 100 == 0:
            losses.append(total.item())

    # Final metrics
    with torch.no_grad():
        total, l_p, l_e, l_s = language_design_loss(proto_vocab, centroid_t, target_dist_t,
                                                      LAM_PROX, LAM_EXPR, LAM_SIMP)
        # Coverage
        sims = F.normalize(proto_vocab, dim=-1) @ centroid_t.T
        coverage = (sims.max(dim=0).values > 0.3).float().mean().item()

    results.append({
        'N_proto': N_proto,
        'L_total': round(total.item(), 4),
        'L_prox':  round(l_p.item(), 4),
        'L_expr':  round(l_e.item(), 4),
        'L_simp':  round(l_s.item(), 4),
        'coverage': round(coverage, 4),
    })
    print('coverage={:.3f} L_total={:.4f}'.format(coverage, total.item()))

import pandas as pd
df = pd.DataFrame(results)
print()
print(df.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# Panel 1: Trade-off curve
ax = axes[0]
ax.plot(df['N_proto'], df['coverage'], 'o-', color='#1D9E75', linewidth=2, label='Coverage')
ax2 = ax.twinx()
ax2.plot(df['N_proto'], df['L_total'], 's--', color='#E24B4A', linewidth=2, label='Total loss')
ax.set_xlabel('Proto-vocabulary size N'); ax.set_ylabel('Coverage (fraction of concepts)', color='#1D9E75')
ax2.set_ylabel('Total design loss', color='#E24B4A')
ax.set_title('Coverage vs vocabulary size\n(increasing N improves expressivity)')
lines1, lbls1 = ax.get_legend_handles_labels()
lines2, lbls2 = ax2.get_legend_handles_labels()
ax.legend(lines1+lines2, lbls1+lbls2, fontsize=9)
ax.grid(True, alpha=0.3)

# Panel 2: Loss breakdown
ax = axes[1]
x = np.arange(len(df))
w = 0.25
ax.bar(x-w, df['L_prox']*LAM_PROX, w, label='L_prox (proximity)', color='#378ADD')
ax.bar(x,   df['L_expr']*LAM_EXPR, w, label='L_expr (expressivity)', color='#1D9E75')
ax.bar(x+w, df['L_simp']*LAM_SIMP, w, label='L_simp (simplicity)', color='#EF9F27')
ax.set_xticks(x); ax.set_xticklabels(['N={}'.format(n) for n in df['N_proto']])
ax.set_ylabel('Weighted loss component'); ax.set_title('Loss decomposition by term\n(PDF: 3-term design loss)')
ax.legend(fontsize=9); ax.grid(axis='y', alpha=0.3)

# Panel 3: Pareto — coverage vs N_proto (the efficiency question)
ax = axes[2]
ax.scatter(df['N_proto'], df['coverage'], s=100, c=df['L_total'], cmap='RdYlGn_r', zorder=3)
for _, row in df.iterrows():
    ax.annotate('N={}'.format(int(row.N_proto)), (row.N_proto, row.coverage),
                xytext=(5,3), textcoords='offset points', fontsize=9)
ax.set_xlabel('Proto-vocabulary size'); ax.set_ylabel('Coverage')
ax.set_title('Coverage vs vocab size\n(colour = total design loss)')
ax.grid(True, alpha=0.3)

plt.suptitle('Experiment C2 — Language-Design Loss\n'
             'Optimising a proto-vocabulary against the PDF 3-term loss function',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('exp_c2_language_design_loss.png', dpi=150, bbox_inches='tight')
plt.show()

print('='*60)
print('EXPERIMENT C2 — SUMMARY')
print('='*60)
print('Loss terms:')
print('  L_proximity:   proto-tokens should be close to centroid space')
print('  L_expressivity: every concept should be near a proto-token')
print('  L_simplicity:  smaller vocabularies preferred')
print()
print('Optimal vocabulary size (coverage > 0.8):')
good = df[df.coverage >= 0.8]
if not good.empty:
    print('  N={:d} achieves coverage={:.3f}'.format(int(good.iloc[0].N_proto), good.iloc[0].coverage))
else:
    best = df.nlargest(1,'coverage').iloc[0]
    print('  Best: N={:d} coverage={:.3f} (increase N_proto range for >0.8)'.format(int(best.N_proto), best.coverage))
print()
print('Key finding: as N grows, L_expr falls and L_simp rises.')
print('The Pareto-optimal vocabulary size is where the slopes balance.')